In [1]:
import pandas as pd
import random
import math

All non-division teams will be played once, and the rest of the games will be made up of division teams. The last number of weeks determined by division_only_end will be division games only. If the total division games is divisible by the number of division teams minus one (how many teams each team would need to play within their division), then each team will play the teams in their division however many times that is. If it is not divisible, All confernece teams will be played a minimum number of games, followed by however many games need to be played to hit the total number of games and each team will not be played the same number of times. For best results, make sure the number of weeks after all non-division games are played is divisible by division teams minus one.

In [ ]:
weeks = 14

# team division lists
division1 = ["Team 1","Team 2","Team 3","Team 4"] # divisions must be equal
division2 = ["Team 5","Team 6","Team 7","Team 8"] # divisions must be equal
division3 = ["Team 9","Team 10","Team 11","Team 12"] #divisions must be equal

# time between teams playing each other a second time
buffer = 2

# the number of weeks that are division-only (occuring at the end of the season)
division_only_end = 3


In [ ]:
# define function to determine schedule using total number of weeks, division lists, game buffer, and division-only count

def season_schedule(week_count, division1_teams, division2_teams, division3_teams, game_buffer, division_only):
    # define fillable sets for team schedules, weekly schedules, and potential opponents
    all_teams = division1_teams + division2_teams + division3_teams
    schedule = {w: [] for w in range(1, week_count + 1)}
    team_schedule = {t: [] for t in all_teams}
    potential_opponents = {t: [] for t in all_teams}

    # calculate total number of conference rounds
    non_division_games = len(division1_teams) + len(division2_teams)
    division_games = week_count - non_division_games
    total_division_rounds = math.ceil(division_games / (len(division1_teams) - 1))

    # determine potential opponents for each team
    for team in all_teams:
        potential_opponents[team] = all_teams
        for round in range(0, int(total_division_rounds) - 1):
            if team in division1_teams:
                potential_opponents[team] = potential_opponents[team] + division1_teams # add division1_all_teams all_teams twice to account for X games for in-conference all_teams
                potential_opponents[team] = [i for i in potential_opponents[team] if i != team]
                # print(potential_opponents[team]) # uncomment if you want to check potential opponents calcs
            elif team in division2_teams:
                potential_opponents[team] = potential_opponents[team] + division2_teams # add division2_all_teams all_teams twice to account for X games for in-conference all_teams
                potential_opponents[team] = [i for i in potential_opponents[team] if i != team]
                # print(potential_opponents[team]) # uncomment if you want to check potential opponents calcs
            else:
                potential_opponents[team] = potential_opponents[team] + division3_teams # add division3_all_teams all_teams twice to account for X games for in-conference all_teams
                potential_opponents[team] = [i for i in potential_opponents[team] if i != team]
                # print(potential_opponents[team]) # uncomment if you want to check potential opponents calcs
    
    def get_division(team):
        if team in division1_teams:
            return division1_teams
        elif team in division2_teams:
            return division2_teams
        else:
            return division3_teams
    
    # determine schedule
    games_per_week = len(all_teams) / 2
    division_only_start = week_count - (division_only + 1)
    for week in range(week_count, 0, -1):
        # print("week "+str(week)) # uncomment to track progress if there are errors
        for game in range(0, int(games_per_week)):
            # print("game "+str(game)) # uncomment to track progress if there are errors
            team1 = random.choice(all_teams)
            while team1 in schedule[week]:
                team1 = random.choice(all_teams)
            schedule[week] = schedule[week] + [team1]

            potential_opponent = potential_opponents[team1]
            potential_opponent = [i for i in potential_opponent if i not in schedule[week]]

            start_date = week - game_buffer
            if start_date < 0:
                start_date = 0

            potential_opponent = [i for i in potential_opponent if i not in team_schedule[team1][0:game_buffer]]
    
            # determine division-only games first
            if week >= division_only_start:
                same_division = get_division(team1)
                potential_opponent = [i for i in potential_opponent if i in same_division]
            
            team2 = random.choice(potential_opponent)
            schedule[week] = schedule[week] + [team2]
            team_schedule[team1] = [team2] + team_schedule[team1] # prepend schedule because we are iterating backwards to determine final division games
            team_schedule[team2] = [team1] + team_schedule[team2] # prepend schedule because we are iterating backwards to determine final division games
            potential_opponents[team1].remove(team2)
            potential_opponents[team2].remove(team1)


    return potential_opponents, team_schedule

In [ ]:
for attempt in range(20000000): # try X times (max 30 minutes, might need to rerun)
    try:
        potential_opponents, team_schedule = season_schedule(weeks, division1, division2, division3, buffer, division_only_end)
        print("it worked")
        break # as soon as it works, break out of the loop
    except Exception as e:
        # if attempt % 1000 == 0:  # uncomment with below to troubleshoot errors
        #     print(f"Attempt {attempt} failed: {e}") # uncomment with above to troubleshoot errors
        continue # otherwise, try again
else: # if the loop exited normally, e.g. if all X attempts failed
    print("it didn't work")

Attempt 0 failed: list index out of range
it worked


In [44]:
# run checks

all_teams_check = division1 + division2 + division3
potential_opponents_checks = {t: [] for t in all_teams_check}

# calculate total number of division rounds
non_division_games = len(division1) + len(division2)
division_games = weeks - non_division_games
total_division_rounds = division_games / (len(division1)-1)

for team in all_teams_check:
    potential_opponents_checks[team] = all_teams_check
    for round in range(0,int(total_division_rounds)-1):
        if team in division1:
            potential_opponents_checks[team] = potential_opponents_checks[team] + division1
            potential_opponents_checks[team] = [i for i in potential_opponents_checks[team] if i != team]
            # print(potential_opponents_checks[team])
        elif team in division2:
            potential_opponents_checks[team] = potential_opponents_checks[team] + division2
            potential_opponents_checks[team] = [i for i in potential_opponents_checks[team] if i != team]
            # print(potential_opponents_checks[team])
        else:
            potential_opponents_checks[team] = potential_opponents_checks[team] + division3
            potential_opponents_checks[team] = [i for i in potential_opponents_checks[team] if i != team]
            # print(potential_opponents_checks[team])
    if sorted(potential_opponents_checks[team]) == sorted(team_schedule[team]):
        print("yes")
    else:
        print(team)

yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes
yes


In [45]:
# make schedule table
schedule_table = pd.DataFrame.from_dict(team_schedule)
schedule_table = schedule_table.T
schedule_table

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
Team 1,Team 4,Team 6,Team 2,Team 11,Team 10,Team 3,Team 9,Team 12,Team 8,Team 5,Team 7,Team 3,Team 2,Team 4
Team 2,Team 12,Team 10,Team 1,Team 8,Team 11,Team 9,Team 7,Team 4,Team 6,Team 3,Team 5,Team 4,Team 1,Team 3
Team 3,Team 6,Team 9,Team 12,Team 7,Team 4,Team 1,Team 5,Team 8,Team 11,Team 2,Team 10,Team 1,Team 4,Team 2
Team 4,Team 1,Team 11,Team 7,Team 6,Team 3,Team 8,Team 10,Team 2,Team 5,Team 12,Team 9,Team 2,Team 3,Team 1
Team 5,Team 11,Team 8,Team 10,Team 12,Team 9,Team 6,Team 3,Team 7,Team 4,Team 1,Team 2,Team 7,Team 8,Team 6
Team 6,Team 3,Team 1,Team 11,Team 4,Team 7,Team 5,Team 8,Team 9,Team 2,Team 10,Team 12,Team 8,Team 7,Team 5
Team 7,Team 9,Team 12,Team 4,Team 3,Team 6,Team 11,Team 2,Team 5,Team 10,Team 8,Team 1,Team 5,Team 6,Team 8
Team 8,Team 10,Team 5,Team 9,Team 2,Team 12,Team 4,Team 6,Team 3,Team 1,Team 7,Team 11,Team 6,Team 5,Team 7
Team 9,Team 7,Team 3,Team 8,Team 10,Team 5,Team 2,Team 1,Team 6,Team 12,Team 11,Team 4,Team 10,Team 11,Team 12
Team 10,Team 8,Team 2,Team 5,Team 9,Team 1,Team 12,Team 4,Team 11,Team 7,Team 6,Team 3,Team 9,Team 12,Team 11


In [47]:
# export to file
schedule_table.to_csv("schedule_division.csv")